In [2]:
import ee

ee.Initialize(project='dri-blm')
print(ee.data.getProjectConfig())


{'name': 'projects/dri-blm/config', 'registrationState': 'REGISTERED_NOT_COMMERCIALLY'}


#VBET

In [5]:

# ---- INPUTS ----
points = ee.FeatureCollection('projects/dri-blm/assets/AIM-MESIC/BLM_Natl_AIM_TerrADat_Hub')  # 50k points
ic = ee.ImageCollection('projects/climate-engine-pro/assets/vbet/vbet-imagecollection')

PK_FIELD = 'PrimaryKey'   # adjust to your actual field name
BAND_NAME = 'b1'  # band in the IC images holding 1/2/masked
SCALE = 5                # native resolution of your raster (adjust)
DRIVE_FOLDER = 'BLM_AIM_RW'
N_BATCHES = 5             # 50k / 5 = 10k points per batch (safe for reduceRegions)
EXPORT_NAME = 'VBET'

# ---- BUILD A SINGLE MOSAIC ----
# Since each image covers a different state/region, they shouldn't overlap
# meaningfully — mosaic() collapses the collection into one image so we
# only do ONE reduceRegions call per batch instead of iterating the IC.
# unmask(0) makes masked pixels read as 0 (so they show up as 0 in output
# rather than dropping the property entirely).
mosaic = ic.select([BAND_NAME]).mosaic().unmask(0).rename('value')

# ---- SAMPLE POINTS ----
# reduceRegions with ee.Reducer.first() is the cheapest way to get a single
# pixel value per point. Much faster than .map() over points calling .sample().
def sample_batch(batch_fc):
    sampled = mosaic.reduceRegions(
        collection=batch_fc,
        reducer=ee.Reducer.first().setOutputs(['value']),
        scale=SCALE,
        tileScale=4   # bump if you hit "computation timed out" / memory errors
    )
    # Keep only the two attributes you want
    return sampled.select([PK_FIELD, 'value'], retainGeometry=False)

# ---- BATCH TO AVOID MEMORY LIMITS ----
# 50k points in one reduceRegions call often hits user-memory-limit-exceeded.
# Split using a deterministic hash of the system:index so batches are balanced.
points_indexed = points.map(
    lambda f: f.set('_batch', ee.Number.parse(
        ee.String(f.get('system:index')).slice(-2), 16  # last 2 hex chars -> 0..255
    ).mod(N_BATCHES))
)

tasks = []
for i in range(N_BATCHES):
    batch = points_indexed.filter(ee.Filter.eq('_batch', i))
    result = sample_batch(batch)

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f'{EXPORT_NAME}_batch_{i}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'{EXPORT_NAME}_batch_{i}',
        fileFormat='CSV',
        selectors=[PK_FIELD, 'value']
    )
    task.start()
    tasks.append(task)
    print(f'Started batch {i}: {task.id}')

print(f'\n{len(tasks)} export tasks submitted.')

Started batch 0: DYU3ZPJODOJZPPYOJMHDJDGZ
Started batch 1: JPYYRLBMNDA4FW7ATIJZLMRH
Started batch 2: 6PXAW6DCO7D5WC3YGSRQCB2I
Started batch 3: C5JIJJ5UHZLFTEH55EMUPNGG
Started batch 4: 6KNAFO5M3CNWBGGBFIZRL2K4

5 export tasks submitted.


# NWI

In [4]:

# ---- INPUTS ----
points = ee.FeatureCollection('projects/dri-blm/assets/AIM-MESIC/BLM_Natl_AIM_TerrADat_Hub')  # 50k points
ic = ee.ImageCollection('projects/climate-engine-pro/assets/nwi/nwi-gridded-20250721')

PK_FIELD = 'PrimaryKey'   # adjust to your actual field name
BAND_NAME = 'WETLAND_TYPE'  # band in the IC images holding 1/2/masked
SCALE = 5                # native resolution of your raster (adjust)
DRIVE_FOLDER = 'BLM_AIM_RW'
N_BATCHES = 5             # 50k / 5 = 10k points per batch (safe for reduceRegions)
EXPORT_NAME = 'NWI'

# ---- BUILD A SINGLE MOSAIC ----
# Since each image covers a different state/region, they shouldn't overlap
# meaningfully — mosaic() collapses the collection into one image so we
# only do ONE reduceRegions call per batch instead of iterating the IC.
# unmask(0) makes masked pixels read as 0 (so they show up as 0 in output
# rather than dropping the property entirely).
mosaic = ic.select([BAND_NAME]).mosaic().unmask(0).rename('value')

# ---- SAMPLE POINTS ----
# reduceRegions with ee.Reducer.first() is the cheapest way to get a single
# pixel value per point. Much faster than .map() over points calling .sample().
def sample_batch(batch_fc):
    sampled = mosaic.reduceRegions(
        collection=batch_fc,
        reducer=ee.Reducer.first().setOutputs(['value']),
        scale=SCALE,
        tileScale=4   # bump if you hit "computation timed out" / memory errors
    )
    # Keep only the two attributes you want
    return sampled.select([PK_FIELD, 'value'], retainGeometry=False)

# ---- BATCH TO AVOID MEMORY LIMITS ----
# 50k points in one reduceRegions call often hits user-memory-limit-exceeded.
# Split using a deterministic hash of the system:index so batches are balanced.
points_indexed = points.map(
    lambda f: f.set('_batch', ee.Number.parse(
        ee.String(f.get('system:index')).slice(-2), 16  # last 2 hex chars -> 0..255
    ).mod(N_BATCHES))
)

tasks = []
for i in range(N_BATCHES):
    batch = points_indexed.filter(ee.Filter.eq('_batch', i))
    result = sample_batch(batch)

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f'{EXPORT_NAME}_batch_{i}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'{EXPORT_NAME}_batch_{i}',
        fileFormat='CSV',
        selectors=[PK_FIELD, 'value']
    )
    task.start()
    tasks.append(task)
    print(f'Started batch {i}: {task.id}')

print(f'\n{len(tasks)} export tasks submitted.')

Started batch 0: 5WRZ4LQRPLRAQX6N6CAZTLKP
Started batch 1: XAX5EP4FSX6OHHGH5K56PY6L
Started batch 2: DYAZGKQIGOGSVG2DYIZ3G2ZZ
Started batch 3: P5CXRU47E5BLUEKAVSS5W3GM
Started batch 4: QZBOXIUXUOWO7AUYJQD74QQS

5 export tasks submitted.
